# Phase 3 — Kaggle Inference (vLLM Few-Shot, No LoRA)

Runs 3 pretrained SLMs with vLLM using few-shot prompting. No LoRA adapters.
Combines predictions via character-level majority voting.

| Model | Size | tp |
|-------|------|----|
| `Qwen/Qwen3-1.7B` | 1.7B | 1 |
| `Qwen/Qwen3-8B` | 8B | 2 |
| `LiquidAI/LFM2.5-1.2B-Instruct` | 1.2B | 1 |

**Hardware**: T4 x2 on Kaggle (internet OFF)

| | |
|---|---|
| **Wheels** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels` |
| **Qwen3-1.7B** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b` |
| **Qwen3-8B** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-8b/models/qwen3_8b` |
| **LFM2.5-1.2B** | `/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-lfm2-5-1-2b/models/lfm2_5_1_2b` |
| **Output** | `/kaggle/working/submission.csv` |

In [ ]:
import os
import sys
import shutil
import subprocess
import site
from pathlib import Path

# =========================
# CONFIG
# =========================
WHEELS = "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-pip-wheel-v2/wheels"
PKG_DIR = "/kaggle/working/pkgs"

# Same package specs — pip finds the matching wheels in WHEELS
PACKAGES = [
    "vllm==0.17.1",
    "transformers==4.56.0",
    "rapidfuzz>=3.0.0",
    "protobuf<6",
    "huggingface-hub>=0.34.0,<1.0",
    "msgspec>=0.18.0",
    "peft>=0.15.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.45.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links", WHEELS,
    *PACKAGES,
])

# =========================
# VERIFY
# =========================
import torch
import transformers
import vllm
import rapidfuzz
import huggingface_hub
import msgspec
import google.protobuf

def where(mod):
    """Show where a module is loaded from."""
    return getattr(mod, "__file__", "builtin/namespace")

print("\n==== VERSION CHECK ====")
print(f"torch:            {torch.__version__}")
print(f"transformers:     {transformers.__version__}  ({where(transformers)})")
print(f"vllm:             {vllm.__version__}  ({where(vllm)})")
print(f"rapidfuzz:        {rapidfuzz.__version__}  ({where(rapidfuzz)})")
print(f"huggingface_hub:  {huggingface_hub.__version__}  ({where(huggingface_hub)})")
print(f"msgspec:          {msgspec.__version__}  ({where(msgspec)})")
print(f"protobuf:         {google.protobuf.__version__}  ({where(google.protobuf)})")

# Quick check: protobuf should now load from PKG_DIR
proto_path = where(google.protobuf)
if PKG_DIR not in proto_path:
    print(f"\n⚠ WARNING: protobuf still loading from outside PKG_DIR: {proto_path}")
else:
    print(f"\n✓ protobuf correctly loading from PKG_DIR")

print("\n==== CUDA ====")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.1f} GB | CC {p.major}.{p.minor}")

In [ ]:
import contextlib, gc, json, logging, re, sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from rapidfuzz.fuzz import partial_ratio_alignment
from tqdm import tqdm
from transformers import AutoTokenizer

from vllm import LLM, SamplingParams
from vllm.config import AttentionConfig
from vllm.v1.attention.backends.registry import AttentionBackendEnum
# GuidedDecodingParams removed in vLLM 0.12 → replaced by StructuredOutputsParams
from vllm.sampling_params import StructuredOutputsParams

try:
    from vllm.distributed.parallel_state import destroy_model_parallel
except ImportError:
    def destroy_model_parallel(): pass

print(f"vLLM version: {__import__('vllm').__version__}")

# T4 CC=7.5 — no native bfloat16
_DTYPE = torch.float16

CONFIG = {
    "DATA_DIR":              Path("/kaggle/input/competitions/nbme-score-clinical-patient-notes"),
    "OUTPUT_DIR":            Path("/kaggle/working"),
    "GPU_MEM_UTIL":          0.90,
    "MAX_MODEL_LEN":         1024,
    "MAX_NEW_TOKENS":        128,
    "LLM_TEMPERATURE":       0.0,
    "MAX_SPANS_PER_FEATURE": 10,
    "VOTE_THRESHOLD":        2,
    "FUZZY_SCORE_CUTOFF":    70.0,
    "SEED":                  42,
}

MODEL_REGISTRY = [
    {
        "name":              "qwen3_1_7b",
        "model_path":        "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-1-7b/models/qwen3_1_7b",
        "vllm_dtype":        "half",
        "tp":                1,
        "enable_thinking":   False,
        "trust_remote_code": False,
    },
    {
        "name":              "qwen3_8b",
        "model_path":        "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-qwen3-8b/models/qwen3_8b",
        "vllm_dtype":        "half",
        "tp":                2,
        "enable_thinking":   False,
        "trust_remote_code": False,
    },
    {
        "name":              "lfm2_5_1_2b",
        "model_path":        "/kaggle/input/notebooks/natbrian/0-kaggle-setup-nbme-score-clinical-lfm2-5-1-2b/models/lfm2_5_1_2b",
        "vllm_dtype":        "half",
        "tp":                1,
        "enable_thinking":   None,
        "trust_remote_code": True,
    },
]

# ── Few-shot examples ──────────────────────────────────────────────────────────
FEW_SHOT_EXAMPLES = [
    {
        "note":    "68 yo male with a 30-pack-year smoking history presents with hemoptysis and a 15-lb weight loss over 3 months.",
        "feature": "smoking history",
        "output":  '{"spans": ["30-pack-year smoking history"]}',
    },
    {
        "note":    "Patient is a 45 yo female with hypertension and type 2 diabetes mellitus. She takes metformin and lisinopril daily. She denies chest pain but reports occasional shortness of breath on exertion.",
        "feature": "current medications",
        "output":  '{"spans": ["metformin", "lisinopril"]}',
    },
    {
        "note":    "32 yo male presents with 3-day history of fever, productive cough with yellowish sputum, and left-sided pleuritic chest pain. Exam reveals decreased breath sounds at left base and dullness to percussion.",
        "feature": "pleuritic chest pain",
        "output":  '{"spans": ["left-sided pleuritic chest pain"]}',
    },
    {
        "note":    "55 yo woman with a history of rheumatoid arthritis managed with methotrexate. She presents for routine follow-up. No joint swelling noted today. Labs show WBC 3.2 and mild transaminase elevation.",
        "feature": "family history of autoimmune disease",
        "output":  '{"spans": []}',
    },
    {
        "note":    "Patient reports dull, aching pain in the right upper quadrant that worsens after fatty meals. She also notes nausea and one episode of vomiting. No jaundice.",
        "feature": "pain characteristics",
        "output":  '{"spans": ["dull, aching pain in the right upper quadrant", "worsens after fatty meals"]}',
    },
]

SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature.\n"
    "Rules:\n"
    "  1. Copy text character-for-character from the note — do NOT paraphrase or rephrase.\n"
    "  2. Only include spans that are literally present in the note.\n"
    "  3. If the feature is absent from the note, return an empty list.\n"
    "  4. Output ONLY valid JSON — no markdown, no explanation, no extra text.\n"
    'Format: {"spans": ["exact text 1", "exact text 2"]}'
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)
print("✓ CONFIG, MODEL_REGISTRY loaded")
print(f"  Models: {[m['name'] for m in MODEL_REGISTRY]}")
print(f"  Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")

## Section 1 — Per-Note Regex FSM Constraint

In [ ]:
def _build_char_class(note_chars: set) -> str:
    parts = []
    for ch in sorted(note_chars, key=ord):
        code = ord(ch)
        if code < 0x20 or code == 0x7F: continue
        if ch == ']':    parts.append(r'\]')
        elif ch == '^':  parts.append(r'\^')
        elif ch == '-':  parts.append(r'\-')
        elif ch == '\\': parts.append(r'\\')
        else:            parts.append(ch)
    return '[' + ''.join(parts) + ']' if parts else r'[^\n]'


def build_constraint_regex(pn_history: str, max_spans: int = 10) -> str:
    note_chars = set(pn_history) - {'"', '\\'}
    char_class = _build_char_class(note_chars)
    span_item  = f'"{char_class}*"'
    additional = r'(?:, ' + span_item + r'){0,' + str(max_spans - 1) + r'}'
    opt_list   = r'(?:' + span_item + additional + r')?'
    return r'\{"spans": \[' + opt_list + r'\]}'

print("✓ Section 1: build_constraint_regex defined")

## Section 2 — Few-Shot Prompt Builder

In [ ]:
def build_few_shot_prompt(feature_text: str, pn_history: str, tokenizer,
                          enable_thinking=None) -> str:
    """
    Build a chat prompt with few-shot examples followed by the target query.
    Alternates user/assistant turns so the model sees the expected output format.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Inject few-shot examples as alternating user/assistant turns
    for ex in FEW_SHOT_EXAMPLES:
        user_content = (
            f'Note: "{ex["note"]}"\n'
            f'Feature: {ex["feature"]}'
        )
        if enable_thinking is False:
            user_content += "\n\n/no_think"
        messages.append({"role": "user",      "content": user_content})
        messages.append({"role": "assistant", "content": ex["output"]})

    # Actual query
    query_content = (
        f'Note: "{pn_history.strip()}"\n'
        f'Feature: {feature_text}'
    )
    if enable_thinking is False:
        query_content += "\n\n/no_think"
    messages.append({"role": "user", "content": query_content})

    kwargs = dict(tokenize=False, add_generation_prompt=True)
    if enable_thinking is not None:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
        except TypeError:
            pass
    return tokenizer.apply_chat_template(messages, **kwargs)

print("✓ Section 2: build_few_shot_prompt defined")

## Section 3 — vLLM Engine Lifecycle

In [ ]:
def init_engine(model_path: str, model_spec: dict, cfg: dict) -> LLM:
    attn_cfg = AttentionConfig(backend=AttentionBackendEnum.TRITON_ATTN)
    log.info(f"  [{model_spec['name']}] Initialising vLLM engine (tp={model_spec['tp']}) ...")
    llm = LLM(
        model                  = model_path,
        dtype                  = model_spec["vllm_dtype"],
        tensor_parallel_size   = model_spec["tp"],
        gpu_memory_utilization = cfg["GPU_MEM_UTIL"],
        max_model_len          = cfg["MAX_MODEL_LEN"],
        enforce_eager          = True,
        trust_remote_code      = model_spec.get("trust_remote_code", False),
        seed                   = cfg["SEED"],
        attention_config       = attn_cfg,
    )
    log.info(f"  [{model_spec['name']}] vLLM engine ready.")
    return llm


def destroy_engine(llm, model_name: str) -> None:
    log.info(f"  [{model_name}] Destroying vLLM engine ...")
    destroy_model_parallel()
    with contextlib.suppress(Exception):
        torch.distributed.destroy_process_group()
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
        total_gb = torch.cuda.mem_get_info()[1] / 1024**3
        log.info(f"  [{model_name}] VRAM after cleanup: {free_gb:.1f}/{total_gb:.1f} GB free")

print("✓ Section 3: init_engine, destroy_engine defined")

## Section 4 — vLLM Inference Runner

In [ ]:
def _parse_json_output(raw_text: str) -> list:
    raw_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL).strip()
    try:
        parsed = json.loads(raw_text)
        return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
    except (json.JSONDecodeError, AttributeError):
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group())
                return [s.strip() for s in parsed.get("spans", []) if isinstance(s, str) and s.strip()]
            except json.JSONDecodeError:
                pass
    return []


def run_inference_vllm(llm, test_rows, pn_map, feat_map, tokenizer,
                       cfg, model_spec) -> list:
    model_name      = model_spec["name"]
    enable_thinking = model_spec.get("enable_thinking")
    log.info(f"  [{model_name}] Building few-shot prompts ...")
    prompts, params_list = [], []

    for _, row in test_rows.iterrows():
        pn_history   = pn_map.get(row["pn_num"], "").replace("\n", " ").strip()
        feature_text = feat_map.get((row["case_num"], row["feature_num"]), "")
        prompts.append(build_few_shot_prompt(feature_text, pn_history, tokenizer, enable_thinking))
        regex  = build_constraint_regex(pn_history, cfg["MAX_SPANS_PER_FEATURE"])
        # backend is set internally by vLLM 0.17.1 — do not pass to constructor
        params_list.append(SamplingParams(
            temperature=cfg["LLM_TEMPERATURE"],
            max_tokens=cfg["MAX_NEW_TOKENS"],
            structured_outputs=StructuredOutputsParams(regex=regex),
        ))

    log.info(f"  [{model_name}] Running vLLM inference on {len(prompts)} rows ...")
    outputs   = llm.generate(prompts=prompts, sampling_params=params_list)
    all_spans = []
    for output in tqdm(outputs, desc=f"  [{model_name}] Parsing", leave=False):
        raw = output.outputs[0].text.strip() if output.outputs else ""
        all_spans.append(_parse_json_output(raw))

    n_nonempty = sum(1 for s in all_spans if s)
    log.info(f"  [{model_name}] Done — non-empty: {n_nonempty}/{len(all_spans)}")
    return all_spans

print("✓ Section 4: _parse_json_output, run_inference_vllm defined")

## Section 5 — Character-Level Majority Voting

In [ ]:
def spans_to_char_array(span_locations: list, note_len: int) -> np.ndarray:
    arr = np.zeros(note_len, dtype=np.uint8)
    for start, end in span_locations:
        arr[max(0, start):min(note_len, end)] = 1
    return arr


def char_array_to_spans(arr: np.ndarray) -> list:
    spans, n, i = [], len(arr), 0
    while i < n:
        if arr[i] == 1:
            start = i
            while i < n and arr[i] == 1: i += 1
            spans.append((start, i))
        else:
            i += 1
    return spans


def locate_span_in_note(span_text: str, pn_history: str,
                        score_cutoff: float = 70.0) -> Optional[tuple]:
    span_text = span_text.strip()
    if not span_text or not pn_history: return None
    idx = pn_history.find(span_text)
    if idx != -1: return (idx, idx + len(span_text))
    idx = pn_history.lower().find(span_text.lower())
    if idx != -1: return (idx, idx + len(span_text))
    result = partial_ratio_alignment(span_text, pn_history, score_cutoff=score_cutoff)
    if result is not None: return (result.dest_start, result.dest_end)
    return None


def character_level_majority_vote(model_predictions, test_rows, pn_map,
                                  vote_threshold=2, fuzzy_cutoff=70.0) -> list:
    n_models, n_rows = len(model_predictions), len(test_rows)
    log.info(f"Majority vote ({n_models} models, threshold={vote_threshold}/{n_models}) ...")
    final_spans = []

    for seq_idx, (_, row) in enumerate(tqdm(test_rows.iterrows(), total=n_rows, desc="Majority vote")):
        pn_history = pn_map.get(row["pn_num"], "")
        note_len   = len(pn_history)
        if note_len == 0:
            final_spans.append([]); continue

        vote_array = np.zeros(note_len, dtype=np.int8)
        for preds in model_predictions:
            locs = [loc for text in preds[seq_idx]
                    if (loc := locate_span_in_note(text, pn_history, fuzzy_cutoff)) is not None]
            if locs:
                vote_array += spans_to_char_array(locs, note_len)

        consensus = (vote_array >= vote_threshold).astype(np.uint8)
        for i, ch in enumerate(pn_history):
            if ch in (' ', '\t', '\n', '\r') and consensus[i]:
                is_start = (i == 0 or consensus[i-1] == 0)
                is_end   = (i == note_len-1 or consensus[i+1] == 0)
                if is_start or is_end: consensus[i] = 0

        final_spans.append(char_array_to_spans(consensus))

    log.info(f"Vote complete — non-empty: {sum(1 for s in final_spans if s)}/{n_rows}")
    return final_spans

print("✓ Section 5: majority vote defined")

## Section 6 — Submission Formatter

In [ ]:
def format_location_string(spans: list, pn_history: str) -> str:
    if not spans: return ""
    clean = []
    for start, end in sorted(spans):
        while start < end and pn_history[start] in (' ', '\t', '\n', '\r'): start += 1
        while end > start and pn_history[end-1] in (' ', '\t', '\n', '\r'): end -= 1
        if start < end: clean.append((start, end))
    merged = []
    for start, end in sorted(clean):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return ";".join(f"{s} {e}" for s, e in merged) if merged else ""


def build_submission(final_spans: list, test_df: pd.DataFrame, pn_map: dict) -> pd.DataFrame:
    rows = []
    for row_idx, (_, test_row) in enumerate(test_df.iterrows()):
        pn_history = pn_map.get(test_row["pn_num"], "")
        spans      = final_spans[row_idx] if row_idx < len(final_spans) else []
        location   = format_location_string(spans, pn_history)
        rows.append({"id": test_row["id"], "location": location if location else np.nan})
    return pd.DataFrame(rows)

print("✓ Section 6: format_location_string, build_submission defined")

## Run — Generate Submission

Pipeline per model:
1. Load pretrained model directly into vLLM (no adapter merge)
2. Run batched inference with 5-shot examples + per-note regex constrained decoding
3. Destroy engine

Then: character-level majority vote → `submission.csv`

In [ ]:
def main():
    cfg      = CONFIG
    data_dir = cfg["DATA_DIR"]

    print("\n" + "="*65)
    print("  PHASE 3: Kaggle Inference (vLLM Few-Shot, No LoRA)")
    print(f"  Models: {[m['name'] for m in MODEL_REGISTRY]}")
    print(f"  Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
    print("="*65 + "\n")

    print("▶ Loading test data ...")
    test_df  = pd.read_csv(data_dir / "test.csv")
    pn_df    = pd.read_csv(data_dir / "patient_notes.csv")
    feat_df  = pd.read_csv(data_dir / "features.csv")
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = feat_df.set_index(["case_num", "feature_num"])["feature_text"].to_dict()
    print(f"  Test rows: {len(test_df)}")

    all_model_predictions = []

    for i, model_spec in enumerate(MODEL_REGISTRY):
        model_name = model_spec["name"]
        model_path = model_spec["model_path"]
        print(f"\n{'='*65}")
        print(f"  Model {i+1}/{len(MODEL_REGISTRY)}: {model_name}  (tp={model_spec['tp']})")
        print(f"  Path: {model_path}")
        print(f"{'='*65}")

        # use_fast=True avoids slow tokenizer that requires vocab.json
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            use_fast=True,
            trust_remote_code=model_spec.get("trust_remote_code", False),
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id

        llm = init_engine(model_path, model_spec, cfg)
        model_spans = run_inference_vllm(llm, test_df, pn_map, feat_map,
                                         tokenizer, cfg, model_spec)
        all_model_predictions.append(model_spans)

        destroy_engine(llm, model_name)
        del llm, tokenizer
        gc.collect()

    print("\n▶ Running character-level majority vote ...")
    effective_threshold = min(cfg["VOTE_THRESHOLD"], len(all_model_predictions))
    final_spans = character_level_majority_vote(
        all_model_predictions, test_df, pn_map,
        vote_threshold=effective_threshold,
        fuzzy_cutoff=cfg["FUZZY_SCORE_CUTOFF"],
    )

    submission_df = build_submission(final_spans, test_df, pn_map)
    out_path = cfg["OUTPUT_DIR"] / "submission.csv"
    submission_df.to_csv(out_path, index=False)

    print("\n" + "="*65)
    print(f"  ✓ Submission saved → {out_path}")
    print(f"  Shape: {submission_df.shape}")
    print(f"  Non-empty: {submission_df['location'].notna().sum()} / {len(submission_df)}")
    print("="*65)
    print(submission_df.head(10).to_string())

main()